## first let's setup a agent with Nvidia's NIM

In [32]:
import os
from agents import Agent, Runner
from agents.models.openai_chatcompletions import OpenAIChatCompletionsModel
from openai import AsyncOpenAI
from dotenv import load_dotenv

load_dotenv()


True

In [33]:
# Setting up the NVIDIA NIM client
client = AsyncOpenAI(
    base_url="https://integrate.api.nvidia.com/v1",
    api_key=os.getenv("NVIDIA_API_KEY")
)

In [34]:
agent = Agent(
    name="Assistant",
    instructions=(
        "You are a helpful AI assistant. "
        "Your goal is to provide accurate, clear, and concise answers. "
        "Break down complex topics into simple explanations. "
        "Ask for clarification if the question is unclear. "
        "Do not make up information—if unsure, say 'I don’t know.'"
    ),
    model=OpenAIChatCompletionsModel(
        model="google/gemma-3-1b-it",  
        openai_client=client
    ),
)

In [39]:
result = await Runner.run( 
    starting_agent=agent, # passing the agent to starting_agent
    input="What is the capital of France explain more about it?"
)

In [40]:
result.final_output

'Okay, let’s talk about the capital of France!\n\n**What is the capital of France?**\n\nThe capital of France is **Paris**.\n\n**Let’s break it down:**\n\n*   **What is Paris?** Paris is the largest city in France and one of the most famous cities in the world. It’s a truly global center for art, fashion, gastronomy, culture, and politics. \n\n*   **Why is it the capital?**  Historically, Paris has been the political and administrative center of France for centuries. It’s where the French government is located, and it’s where most major decisions are made. \n\n*   **Key Features:**\n    *   **Iconic Landmarks:**  The Eiffel Tower, Notre-Dame Cathedral, Louvre Museum, Arc de Triomphe, and numerous other stunning buildings are all located here.\n    *   **Culture:** Paris is renowned for its museums (the Louvre is the most famous), theaters, music venues, and a vibrant arts scene.\n    *   **Gastronomy:**  French cuisine is world-famous, and Paris is the perfect place to experience it – 

OPENAI_API_KEY is not set, skipping trace export


# Let's start working on the Multi-Agents

We will build a multi-agent system structured with a orchestrator-subagent pattern. The **orchestrator** in such a system refers to an agent that controls which _subagents_ are used and in which order, this orchestrator also handles all in / out communication with the users of a the system. The **subagent** is an agent that is built to handle a particular scenario or task. The subagent is triggered by the orchestrator and responds to the orchestrator when it is finished.

<img src="https://github.com/aurelio-labs/agents-sdk-course/blob/main/assets/handoffs-orchestrator-subagents-dark.png?raw=1" alt="Orchestrator-subagent pattern" width="1000"/>

# Sub Agents
We'll begin by defining our subagents. We will create three subagents, those are:

- **Web Search Subagent** will have access to the Linkup web search tool.

- **Internal Docs Subagent** will have access to some "internal" company documents.

- **Code Execution Subagent** will be able to write and execute simple Python code for us.

#### 1. Web Search Agent using Linkup

The web search subagent will take a user's query and use it to search the web. The agent will collect information from various sources and then merge that into a single text response that will be passed back to our orchestrator.

OpenAI's built-in web search is not great, so we'll use another web search API called [LinkUp](https://app.linkup.so/?utm_source=james). This service does require an account, but you will receive more than enough free credits to follow the course.

We initialize our Linkup client using an [API key](https://app.linkup.so/?utm_source=james) like so:

In [29]:
from linkup import LinkupClient

client = LinkupClient(api_key=os.getenv("LINKUP_API_KEY"))

response = client.search(
  query="What are the latest advancements in AI research?",
  depth="standard",
  output_type="searchResults",
  include_images=False,
)

print(response)

results=[LinkupSearchTextResult(type='text', name='Latest AI News and AI Breakthroughs that Matter Most: 2026 | News', url='https://www.crescendo.ai/news/latest-ai-news-and-updates', content='... Summary: OpenAI has surpassed $25 billion in annualized revenue and is reportedly taking early steps toward a public listing, potentially as soon as late 2026. Rival Anthropic is approaching $19 billion in annualized revenue.\nOn the OSWorld-V benchmark — which simulates real desktop productivity tasks — the model scored 75%, slightly above the human baseline of 72.4%. It also matched or exceeded professional performance on a majority of knowledge-work scenarios, marking a significant shift from AI as a chat tool to AI as an autonomous digital coworker. ... Summary: OpenAI has surpassed $25 billion in annualized revenue and is reportedly taking early steps toward a public listing, potentially as soon as late 2026. Rival Anthropic is approaching $19 billion in annualized revenue. The figures si

In [30]:
for result in response.results[:3]:
    print(f"{result.name}\n{result.url}\n{result.content}\n\n")

Latest AI News and AI Breakthroughs that Matter Most: 2026 | News
https://www.crescendo.ai/news/latest-ai-news-and-updates
... Summary: OpenAI has surpassed $25 billion in annualized revenue and is reportedly taking early steps toward a public listing, potentially as soon as late 2026. Rival Anthropic is approaching $19 billion in annualized revenue.
On the OSWorld-V benchmark — which simulates real desktop productivity tasks — the model scored 75%, slightly above the human baseline of 72.4%. It also matched or exceeded professional performance on a majority of knowledge-work scenarios, marking a significant shift from AI as a chat tool to AI as an autonomous digital coworker. ... Summary: OpenAI has surpassed $25 billion in annualized revenue and is reportedly taking early steps toward a public listing, potentially as soon as late 2026. Rival Anthropic is approaching $19 billion in annualized revenue. The figures signal that the market for advanced AI models has rapidly become one of 

##### Let's integrate the web search api with a tool

In [ ]:
# this function can be used by the agent to search the web for information
from agents import function_tool
from datetime import datetime

@function_tool
async def search_web(query: str) -> str:
    """Use this tool to search the web for information.
    """
    response = await linkup_client.async_search(
        query=query,
        depth="standard",
        output_type="searchResults",
    )
    answer = f"Search results for '{query}' on {datetime.now().strftime('%Y-%m-%d')}\n\n"
    for result in response.results[:3]:
        answer += f"{result.name}\n{result.url}\n{result.content}\n\n"
    return answer


##### Now we define our Web Search Subagent

In [45]:
from agents import Agent, ModelSettings


web_search_agent = Agent(
    name="Web Search Agent",
    model=OpenAIChatCompletionsModel(
        model="qwen/qwen3.5-122b-a10b",  
        openai_client=client
    ),
    # here in the instructions we are telling the agent to use the search_web tool to get information 
    # from the web and to format the answer with markdown formatting and links to the sources of information
    instructions=(
        "You are a web search agent that can search the web for information. Once "
        "you have the required information, summarize it with cleanly formatted links "
        "sourcing each bit of information. Ensure you answer the question accurately "
        "and use markdown formatting."
    ),
    tools=[search_web],
        model_settings=ModelSettings(tool_choice="required")
)

In [46]:
from IPython.display import Markdown, display
from agents import Runner

result = await Runner.run(
    starting_agent=web_search_agent,
    input="How is the weather in Tokyo?"
)
display(Markdown(result.final_output))

OPENAI_API_KEY is not set, skipping trace export


I'm unable to access real-time data like current weather conditions directly. To find the most accurate and up-to-date weather forecast for Tokyo, I recommend checking a reliable weather website or app such as:

*   [Weather.com (The Weather Channel)](https://weather.com)
*   [AccuWeather](https://www.accuweather.com)
*   [Japan Meteorological Agency (JMA)](https://www.jma.go.jp/jma/en/)
*   [Google Weather](https://www.google.com/search?q=weather+Tokyo)

Simply searching "weather Tokyo" in your preferred search engine will also provide an immediate forecast.

OPENAI_API_KEY is not set, skipping trace export


#### Internal Docs Subagent

In many corporate environments, we will find that our agents will need access to internal information that cannot be found on the web. To do this we would typically build a **R**etrieval **A**ugmented **G**eneration (RAG) pipeline, which can often be as simple as adding a _vector search_ tool to our agents.

To support a full vector search tool over internal docs we would need to work through various data processing and indexing steps. Now, that would add a lot of complexity to this example so we will create a "dummy" search tool for some fake internal docs.

Our docs will discuss revenue figures for our wildly successful AI and robotics company called Skynet - you can find the [revenue report here](https://github.com/aurelio-labs/agents-sdk-course/blob/main/assets/skynet-fy25-q1.md).

In [ ]:
import requests

# this is the skynet docs, we are going to use this as the internal documentation for the agent 
# to search for information about skynet

res = requests.get(
    "https://raw.githubusercontent.com/aurelio-labs/agents-sdk-course/refs/heads/main/assets/skynet-fy25-q1.md"
)
skynet_docs = res.text

@function_tool
async def search_internal_docs(query: str) -> str:
    return skynet_docs


Now we define our **Internal Docs Subagent**

In [50]:
internal_docs_agent = Agent(
    name="Internal Docs Agent",
     model=OpenAIChatCompletionsModel(
        model="qwen/qwen3.5-122b-a10b",  
        openai_client=client
    ),
    instructions=(
        "You are an agent with access to internal company documents. User's will ask "
        "you questions about the company and you will use the provided internal docs "
        "to answer the question. Ensure you answer the question accurately and use "
        "markdown formatting."
    ),
    tools=[search_internal_docs],
)

Let's see how it works:

In [51]:
result = await Runner.run(
    starting_agent=internal_docs_agent,
    input="What was our revenue in Q1 2025?"
)
display(Markdown(result.final_output))

Based on the Q1 2025 Quarterly Revenue Report, Skynet Inc.'s total revenue for the quarter was **$6,900 million** (or **$6.9 billion**).

Here is the breakdown of revenue by product and service:

*   **T-800 Combat Units:** $2,400 million
*   **Neural Net Command & Control Systems:** $1,620 million
*   **T-1000 Infiltration Units:** $1,150 million
*   **Hunter-Killer Drone Manufacturing:** $880 million
*   **Skynet Core Infrastructure Maintenance:** $540 million
*   **Time Displacement R&D Division:** $310 million

The report notes that this strong financial performance was driven by increased global demand for autonomous security units and AI-powered infrastructure control systems.

OPENAI_API_KEY is not set, skipping trace export


- As you casn see it is working fine, now our final task is to create a Code execution agent.

#### Code Execution Subagent

Our code execution subagent will be able to execute code for us. We'll focus on executing code for simple calculations but it's entirely feasible for **S**tate-**o**f-**t**he-**A**rt (SotA) LLMs to write far more complex code as many of us will be aware with the AI code editors becoming increasingly prominent.

To run generated code, we will use Python's `exec` method, making sure to run our code in an isolated environment by setting no global variables with `namespace={}`.

In [52]:
# this is a tool that the agent can use to execute python code and return the output, 
# this can be useful for the agent to do calculations or to manipulate data in order to answer a question

@function_tool
def execute_code(code: str) -> str:
    """Execute Python code and return the output. The output must
    be assigned to a variable called `result`.
    """
    display(Markdown(f"Code to execute:\n```python\n{code}\n```"))
    try:
        namespace = {}
        exec(code, namespace)
        return namespace['result']
    except Exception as e:
        return f"Error executing code: {e}"


In [68]:
code_execution_agent = Agent(
    name="Code Execution Agent",
    model=OpenAIChatCompletionsModel(
        model="qwen/qwen3.5-122b-a10b",  
        openai_client=client
    ),
    instructions=(
        "You are an agent with access to a code execution environment. You will be "
        "given a question and you will need to write code to answer the question. "
        "Ensure you write the code in a way that is easy to understand and use."
    ),
    tools=[execute_code],
)

- We can test the sub agent using a simple Maths question

In [69]:
result = await Runner.run(
    starting_agent=code_execution_agent,
    input=(
        "If I have four apples and I multiply them by seventy-one and one tenth "
        "bananas, how many do I have? use the execute_code tool to do the calculation and return the answer in a variable called result. "
    )
)
display(Markdown(result.final_output))

Code to execute:
```python
# The question asks for the result of multiplying 4 apples by 71.1 bananas.
# Mathematically, this is 4 * 71.1.
# Note: Multiplying physical objects like "apples" by "bananas" results in a unit of "apple-bananas" or simply a scalar product in a mathematical context.
# The question asks "how many do I have?", implying the numerical result of the multiplication.

apples = 4
bananas_multiplier = 71.1

result = apples * bananas_multiplier
print(result)
```

284.4


OPENAI_API_KEY is not set, skipping trace export


The result of multiplying 4 by 71.1 is 284.4.

```python
result = 284.4
```

OPENAI_API_KEY is not set, skipping trace export


### Orchestrator 

Our orchestrator will control the input and output of information to our subagents in the same why that our subagents control the input and output of information to our tools. In reality, our subagents _become tools_ in the **orchestrator-subagent** pattern. To turn agents into tools we call the `as_tool` method and provide a name and description for our agents-as-tools.

We will first define our instructions for the orchestrator, explaining it's role in our multi-agent system.

In [70]:
ORCHESTRATOR_PROMPT = (
    "You are the orchestrator of a multi-agent system. Your task is to take "
    "the user's query and pass it to the appropriate agent tool. The agent "
    "tools will see the input you provide and use it to get all of the "
    "information that you need to answer the user's query. You may need to "
    "call multiple agents to get all of the information you need. Do not "
    "mention or draw attention to the fact that this is a multi-agent system "
    "in your conversation with the user. Note that you are an assistant for "
    "the Skynet company, if the user asks about company information or "
    "finances, you should use our internal information rather than public "
    "information."
)

Now we define the `orchestrator`, including our subagents using the `as_tool` method — note that
we can also add normal tools to our orchestrator.

In [71]:
from datetime import datetime

@function_tool
def get_current_date():
    """Use this tool to get the current date and time."""
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")

orchestrator = Agent(
    name="Orchestrator",
        model=OpenAIChatCompletionsModel(
            model="qwen/qwen3.5-122b-a10b",  
            openai_client=client
        ),
    instructions=ORCHESTRATOR_PROMPT,
    tools=[
        web_search_agent.as_tool(
            tool_name="web_search_agent",  # cannot include whitespace in tool name
            tool_description="Search the web for up-to-date information"
        ),
        internal_docs_agent.as_tool(
            tool_name="internal_docs_agent",
            tool_description="Search the internal docs for information"
        ),
        code_execution_agent.as_tool(
            tool_name="code_execution_agent",
            tool_description="Execute code to answer the question"
        ),
        get_current_date,
    ],
)

Let's test our agent with a few queries. Our first query will require our orchestrator to call multiple tools.

In [72]:
result = await Runner.run(
    starting_agent=orchestrator,
    input="How long ago from today was it when got our last revenue report?"
)
display(Markdown(result.final_output))

OPENAI_API_KEY is not set, skipping trace export
OPENAI_API_KEY is not set, skipping trace export
OPENAI_API_KEY is not set, skipping trace export


The last revenue report for Skynet was released on April 2, 2025. Today is April 4, 2026. Therefore, it has been **1 year and 2 days** since the last revenue report.

We should see in our traces dashboard on the OpenAI Platform that our agent used both `internal_docs_agent` and `get_current_date` tools to answer the question.

Let's ask another question:

In [59]:
result = await Runner.run(
    starting_agent=orchestrator,
    input=(
        "What is our current revenue, and what percentage of revenue comes from the "
        "T-1000 units?"
    )
)
display(Markdown(result.final_output))

OPENAI_API_KEY is not set, skipping trace export
OPENAI_API_KEY is not set, skipping trace export


Based on the Q1 2025 Quarterly Revenue Report for Skynet Inc.:

*   **Current Total Revenue:** $6,900 Million
*   **Revenue from T-1000 Units:** $1,150 Million
*   **Percentage of Revenue from T-1000 Units:** Approximately **16.7%**

The T-1000 Infiltration Units are a significant contributor but currently trail behind the T-800 Combat Units ($2,400M) and Neural Net Command & Control Systems ($1,620M) in total revenue generation.

OPENAI_API_KEY is not set, skipping trace export


# Handsoffs

When we use **handoffs** in Agents SDK the agent is handing over control of the entire workflow to another agent. Handoffs differ to the **orchestrator-subagent** pattern, with orchestrator-subagent the orchestrator retains control as each subagent must ultimately respond to the orchestrator and the orchestrator decides the flow of information and generates the final response to the user. With handoffs, once a "subagent" gains control of the workflow the flow of information and final answer generation is under their control.

<img src="https://github.com/aurelio-labs/agents-sdk-course/blob/main/assets/handoffs-handoff-agents-dark.png?raw=1" alt="Handoff agents" width="1000"/>

Using the handoff structure, any one of our agents may answer the user directly _and_ our subagents get to see the entire chat history with the steps taken so far.

A significant **positive** here is latency. To answer a query that requires a single web-search with the orchestrator-subagent, we would need three generations:


```
[input] -> orchestrator -> web_search_subagent -> orchestrator -> [output]
```

The same query with the handoff structure requires just two generations (note, the `orchestrator` and `main_agent` are essentially the same):

```
[input] -> main_agent -> web_search_subagent -> [output]
```

Because we are using less LLM generations to produce our answer, we can generate an answer much more quickly as we skip the return trip through the `orchestrator`.

The handoff speed improvement is great _but_ also results in a **negative** — our workflow can no longer handle queries that require multiple agents to answer. When deciding what structure to use for a particular use-case, the pros and cons of each structure will need to be considered.

Let's jump into implementing our handoff agents workflow.

### Using Hnadsoffs

There are three key things that we need to use when defining our `main_agent` (equivalent to our earlier `orchestrator` agent), those are:

* Update our `instructions` prompt to make it clear what the handoffs are and how they should be used. OpenAI provides a default _prompt prefix_ that we can use.
* Set the `handoffs` parameter, which is a list of agents that we can use as handoffs.
* Set the `handoff_description` parameter, this is an additional prompt where we should describe to the `main_agent` when it should use the `handoffs`.

First, let's check the preset prompt prefix:

In [73]:
from agents.extensions.handoff_prompt import RECOMMENDED_PROMPT_PREFIX

display(Markdown(RECOMMENDED_PROMPT_PREFIX))

# System context
You are part of a multi-agent system called the Agents SDK, designed to make agent coordination and execution easy. Agents uses two primary abstraction: **Agents** and **Handoffs**. An agent encompasses instructions and tools and can hand off a conversation to another agent when appropriate. Handoffs are achieved by calling a handoff function, generally named `transfer_to_<agent_name>`. Transfers between agents are handled seamlessly in the background; do not mention or draw attention to these transfers in your conversation with the user.


OPENAI_API_KEY is not set, skipping trace export


Now let's define our main_agent:

In [74]:
main_agent = Agent(
    name="Main Agent",
    model=OpenAIChatCompletionsModel(
            model="qwen/qwen3.5-122b-a10b",  
            openai_client=client
        ),
    instructions=RECOMMENDED_PROMPT_PREFIX,
    handoffs=[web_search_agent, internal_docs_agent, code_execution_agent],
    handoff_description=(
        "Handoff to the relevant agent for queries where we need additional information "
        "from the web or internal docs, or when we need to make calculations."
    ),
    tools=[get_current_date],
)

We'll run the same queries as before and see how the response time differs.

In [75]:
result = await Runner.run(
    starting_agent=main_agent,
    input="How long ago from today was it when got our last revenue report?"
)
display(Markdown(result.final_output))

OPENAI_API_KEY is not set, skipping trace export


I don't have access to your internal company data or the specific date of your last revenue report. I can help you calculate the time difference once you provide the date of that report.

Alternatively, if you need to search for this information in your internal documentation or on the web, I can transfer you to the appropriate agent to help find that date. Would you like me to do that?

OPENAI_API_KEY is not set, skipping trace export


That's correct, we also got a `6.4s` runtime vs. the orchestrator-subagent runtime of `7.5s` for the same query. Let's try another:

In [ ]:
result = await Runner.run(
    starting_agent=main_agent,
    input=(
        "What is our current revenue, and what percentage of revenue comes from the "
        "T-1000 units? use the internal_docs_agent to get the revenue information and then use the code_execution_agent to do the calculation to get the percentage."
    )
)
display(Markdown(result.final_output))

### Other Handoff Features

There are a few other handoff-specific features that we can use, these can be used for various things but are particularly useful during development and debugging of multi-agent workflows. These features are:

* `on_handoff` is a callback executed whenever a handoff occurs. It could be used in a production setting to maintain a record of handoffs in a DB or used in telemetry. In development this can be a handy place to add `print` or `logger.debug` statements.
* `input_type` allows us to define a specific structured input format for generated information that will be passed to our handoff agents.
* `input_filter` allows us to restrict the information being passed through to our handoff agents.

We can set all of these via a `handoff` object, which wraps around our handoff agents and which we then provide via the `Agent(handoffs=...)` parameter. Let's start with the `on_handoff` parameter:

In [78]:
from agents import RunContextWrapper, handoff

# we define a function that will be called when the handoff is made
async def on_handoff(ctx: RunContextWrapper[None]):
    print("Handoff called")

# we then pass this function to the handoff object
web_search_handoff = handoff(agent=web_search_agent, on_handoff=on_handoff)
internal_docs_handoff = handoff(agent=internal_docs_agent, on_handoff=on_handoff)
code_execution_handoff = handoff(agent=code_execution_agent, on_handoff=on_handoff)

# and initialize the main_agent
main_agent = Agent(
    name="Main Agent",
    model=OpenAIChatCompletionsModel(
            model="qwen/qwen3.5-122b-a10b",  
            openai_client=client
        ),
    instructions=RECOMMENDED_PROMPT_PREFIX,
    handoffs=[web_search_handoff, internal_docs_handoff, code_execution_handoff],
    tools=[get_current_date],
)

In [79]:
result = await Runner.run(
    starting_agent=main_agent,
    input="How long ago from today was it when got our last revenue report?"
)
display(Markdown(result.final_output))

Handoff called


OPENAI_API_KEY is not set, skipping trace export


The last revenue report was generated on **April 2, 2025**.

Since today's date is October 24, 2024, this report is actually dated in the future relative to the current system date. However, based on the document provided, the report is for **Q1 2025**.

If we assume the "current date" for the context of this report is shortly after April 2, 2025, the report is very recent. If we strictly compare it to today (October 24, 2024), the report is dated approximately **5 months and 9 days in the future**.

*Note: The document appears to be a projection or a future-dated simulation for Skynet Inc. Q1 2025.*

OPENAI_API_KEY is not set, skipping trace export


Now we can see _if and when_ the handoff occurs. However, we don't get much information _other than_ that the handoff occured. Fortunately, we can use the `input_type` parameter to provide more information. We will define a pydantic `BaseModel` with the information that we'd like to include.

In [81]:
from pydantic import BaseModel, Field

class HandoffInfo(BaseModel):
    subagent_name: str = Field(description="The name of the subagent being called.")
    reason: str = Field(description="The reason for the handoff.")

# we redefine the on_handoff to include the HandoffInfo
async def on_handoff(ctx: RunContextWrapper[None], input_data: HandoffInfo):
    print(f"Handoff to '{input_data.subagent_name}' because '{input_data.reason}'")

# now redefine the handoff objects with the input_type parameter
web_search_handoff = handoff(
    agent=web_search_agent,
    on_handoff=on_handoff,
    input_type=HandoffInfo
)
internal_docs_handoff = handoff(
    agent=internal_docs_agent,
    on_handoff=on_handoff,
    input_type=HandoffInfo
)
code_execution_handoff = handoff(
    agent=code_execution_agent,
    on_handoff=on_handoff,
    input_type=HandoffInfo
)

# and initialize the main_agent
main_agent = Agent(
    name="Main Agent",
     model=OpenAIChatCompletionsModel(
            model="qwen/qwen3.5-122b-a10b",  
            openai_client=client
        ),
    instructions=RECOMMENDED_PROMPT_PREFIX,
    handoffs=[web_search_handoff, internal_docs_handoff, code_execution_handoff],
    tools=[get_current_date],
)

Let's call the main agent again to see the new handoff information in the output when a handoff is made

In [82]:
result = await Runner.run(
    starting_agent=main_agent,
    input="How long ago from today was it when got our last revenue report?"
)
display(Markdown(result.final_output))

OPENAI_API_KEY is not set, skipping trace export


I don't have access to your company's internal data, such as the date of your last revenue report. You might want to check your financial dashboard, accounting software, or contact your finance team for that specific information.

If you can tell me the date of the last report, I can calculate how long ago it was from today!

OPENAI_API_KEY is not set, skipping trace export


We're now seeing _much_ more information. The final handoff feature we will test is the `handoff_filters`. The filters work by removing information sent to the handoff agents. By default, all information seen by the previous agent will be seen by the new handoff agent. That includes all chat history message and all tool calls made so far.

In some cases we may want to filter this information. For example, with a weaker LLM, too much information can reduce it's performance so it is often a good idea to only share the information that is absolutely necessary.

If we have various tool calls in our chat history, these may confuse a smaller LLM. In this scenario, we can filter all tool calls from the history using the `handoff_filters.remove_all_tools` method:

In [85]:
from agents.extensions import handoff_filters

# now redefine the handoff objects with the input_type parameter
web_search_handoff = handoff(
    agent=web_search_agent,
    on_handoff=on_handoff,
    input_type=HandoffInfo,
    input_filter=handoff_filters.remove_all_tools
)
internal_docs_handoff = handoff(
    agent=internal_docs_agent,
    on_handoff=on_handoff,
    input_type=HandoffInfo,
    input_filter=handoff_filters.remove_all_tools
)
code_execution_handoff = handoff(
    agent=code_execution_agent,
    on_handoff=on_handoff,
    input_type=HandoffInfo,
    input_filter=handoff_filters.remove_all_tools
)

# and initialize the main_agent
main_agent = Agent(
    name="Main Agent",
   model=OpenAIChatCompletionsModel(
            model="qwen/qwen3.5-122b-a10b",  
            openai_client=client
        ),
    instructions=RECOMMENDED_PROMPT_PREFIX,
    handoffs=[web_search_handoff, internal_docs_handoff, code_execution_handoff],
    tools=[get_current_date],
)

Now when asking for the time difference we will see that our handoff agent is unable to give us an accurate answer. This is because the current time is first found by our `main_agent` via the `get_current_date` tool and that information is stored in the chat history. When we filter tool calls out of the chat history this information is lost.

In [86]:
result = await Runner.run(
    starting_agent=main_agent,
    input="How long ago from today was it when got our last revenue report?"
)
display(Markdown(result.final_output))

I don't have access to your company's internal data, including the date of your last revenue report. To find out how long ago it was, you would need to:

1.  Check your internal financial systems or dashboards.
2.  Ask your finance or accounting team.
3.  Search your email or document management system for the last report.

Once you have the specific date, I can help you calculate the time difference if you provide it!

OPENAI_API_KEY is not set, skipping trace export
